# Домашнее задание «Функции и работа с данными»

## Задание 1  
Напишите функцию, которая классифицирует фильмы из материалов занятия по правилам:

* оценка 2 и ниже — низкий рейтинг;
* оценка 4 и ниже — средний рейтинг;
* оценка 4.5 и 5 — высокий рейтинг.

Результат классификации запишите в столбец class.

In [40]:
import pandas as pd

In [41]:
ratings = pd.read_csv('ml-latest-small/ratings.csv')

def classify_rating(rating):
    if rating <= 2:
        return 'низкий'
    elif rating <= 4:
        return 'средний'
    else:
        return 'высокий'

In [42]:
ratings['class'] = ratings['rating'].apply(classify_rating)
ratings.head()

,userId,movieId,rating,timestamp,class
0,1,31,2.5,1260759144,средний
1,1,1029,3.0,1260759179,средний
2,1,1061,3.0,1260759182,средний
3,1,1129,2.0,1260759185,низкий
4,1,1172,4.0,1260759205,средний


## Задание 2

Используйте файл `keywords.csv`.

Нужно написать гео-классификатор, который каждой строке сможет выставить географическую принадлежность определённому региону. Т. е. если поисковый запрос содержит название города региона, то в столбце `region` пишется название этого региона. Если поисковый запрос не содержит названия города, то ставим `undefined`.

Правила распределения по регионам Центр, Северо-Запад и Дальний Восток:

```
geo_data = {
'Центр': ['москва', 'тула', 'ярославль'],
'Северо-Запад': ['петербург', 'псков', 'мурманск'],
'Дальний Восток': ['владивосток', 'сахалин', 'хабаровск']
}
```
Результат классификации запишите в отдельный столбец `region`.

In [43]:
keywords = pd.read_csv('ml-latest-small/keywords.csv')
geo_data = {
    'Центр': ['москва', 'тула', 'ярославль'],
    'Северо-Запад': ['петербург', 'псков', 'мурманск'],
    'Дальний Восток': ['владивосток', 'сахалин', 'хабаровск']
}

def classify_region_row(row):
    keyword_lower = row['keyword'].lower()
    for region, cities in geo_data.items():
        for city in cities:
            if city in keyword_lower: return region
    return 'undefined'

keywords['region'] = keywords.apply(classify_region_row, axis=1)
keywords[keywords['region'] != 'undefined'][['keyword', 'region']]

,keyword,region
127,авито москва,Центр
370,авито ру санкт петербург,Северо-Запад
564,погода в санкт петербурге,Северо-Запад
849,авито ярославль,Центр
1063,фарпост владивосток,Дальний Восток
...,...,...
99590,авито ярославль автомобили с пробегом,Центр
99634,северпост новости мурманской области,Северо-Запад
99808,полармед мурманск запись на прием,Северо-Запад
99890,яндекс метро москва,Центр


## Задание 3 (бонусное)

Есть мнение, что раньше снимали настоящее кино, не то что сейчас. Ваша задача — проверить это утверждение, используя файлы с рейтингами фильмов из прошлого домашнего занятия: файл `movies.csv` и `ratings.csv` из базы. Нужно проверить, верно ли, что с ростом года выпуска фильма его средний рейтинг становится ниже.

Вы не будете затрагивать субьективные факторы выставления этих рейтингов, а пройдётесь по алгоритму:

1. В переменную years запишите список из всех годов с 1950 по 2010 года.

2. Напишите функцию `production_year`, которая каждой строке из названия фильма выставляет год выпуска. Не все названия фильмов содержат год выпуска в одинаковом формате, поэтому используйте алгоритм:

* для каждой строки пройдите по всем годам списка `years`;
* если номер года присутствует в названии фильма, то функция возвращает этот год, как год выпуска;
* если ни один из номеров года списка years не встретился в названии фильма, то возвращается 1900 год.
* Запишите год выпуска фильма по алгоритму пункта 2 в новый столбец `year`.

3. Посчитайте средний рейтинг всех фильмов для каждого значения столбца `year` и отсортируйте результат по убыванию рейтинга.

In [44]:
movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')
years = list(range(1950, 2011))

In [45]:
def production_year(title):
    title_str = str(title)
    
    for year in years:
        year_str = str(year)
        if year_str in title_str:
            return year
    
    return 1900

In [46]:
movies['year'] = movies['title'].apply(production_year)

In [47]:
movies['year'].value_counts().sort_index()

year
1900    1610
1950      27
1951      23
1952      27
1953      36
        ... 
2006     261
2007     256
2008     247
2009     254
2010     228
Name: count, Length: 62, dtype: int64

In [48]:
merged = ratings.merge(movies, on='movieId')
avg_rating_by_year = merged.groupby('year')['rating'].mean().reset_index()

In [52]:
avg_rating_by_year_desc = avg_rating_by_year.sort_values('rating', ascending=False)
avg_rating_by_year_desc

,year,rating
8,1957,4.014241
23,1972,4.011136
3,1952,4.000000
5,1954,3.994220
2,1951,3.983539
...,...,...
56,2005,3.448434
54,2003,3.444777
47,1996,3.426600
48,1997,3.415764
